In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

df = pd.read_csv("/creditcard.csv")

print("Dataset shape:", df.shape)
print(df["Class"].value_counts())

Dataset shape: (11959, 31)
Class
0.0    11906
1.0       52
Name: count, dtype: int64


In [3]:
df = df.dropna(subset=["Class"])

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (9566, 30)
Testing data: (2392, 30)


In [4]:
from imblearn.under_sampling import RandomUnderSampler

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(sampling_strategy=0.1, random_state=42)),
    ("under", RandomUnderSampler(sampling_strategy=0.5, random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


In [5]:

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\nKNN Results (k=5)")
print("Accuracy:", round(accuracy, 4))
print("F1-Score:", round(f1, 4))

param_grid = {
    "knn__n_neighbors": [3, 5, 7]
}
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)




KNN Results (k=5)
Accuracy: 0.9921
F1-Score: 0.4571


In [6]:
# best k find karne ke liye model tune kar rahe hain
grid_search.fit(X_train, y_train)



Fitting 3 folds for each of 3 candidates, totalling 9 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote',
                                        SMOTE(random_state=42,
                                              sampling_strategy=0.1)),
                                       ('under',
                                        RandomUnderSampler(random_state=42,
                                                           sampling_strategy=0.5)),
                                       ('knn',
                                        KNeighborsClassifier(n_jobs=-1))]),
             n_jobs=-1, param_grid={'knn__n_neighbors': [3, 5, 7]},
             scoring='f1', verbose=1)

In [7]:
print("\nBest Parameters:", grid_search.best_params_)
print("Best CV F1-Score:", round(grid_search.best_score_, 4))




Best Parameters: {'knn__n_neighbors': 3}
Best CV F1-Score: 0.646


In [8]:
# best model se prediction kar rahe hain
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

# tuned model ki performance check kar rahe hain
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned, zero_division=0)

print("\nTuned KNN Results")
print("Accuracy:", round(tuned_accuracy, 4))
print("F1-Score:", round(tuned_f1, 4))




Tuned KNN Results
Accuracy: 0.9941
F1-Score: 0.5625


In [9]:
# dono models ke results compare kar rahe hain
results = pd.DataFrame({
    "Model": ["KNN (k=5)", "Tuned KNN"],
    "Accuracy": [accuracy, tuned_accuracy],
    "F1-Score": [f1, tuned_f1]
})

print("\nModel Comparison")
print(results.to_string(index=False))


Model Comparison
    Model  Accuracy  F1-Score
KNN (k=5)  0.992057  0.457143
Tuned KNN  0.994147  0.562500
